# 01. 데이터 전처리

**처리 단계**
1. 이미지 매치 실패 채널 제거
2. 피처 데이터 조인 (채널 피처, 이미지 스코어)
3. 채널 정보 병합
4. 피처 변환 (duration → 초, published_at → 경과일)
5. 조회수 필터링 (0 / 100 미만 / 1,000만 이상)
6. 클립채널/부계정 제거
7. 활동 중단 채널 제거 (6개월 기준)
8. 2022년 이전 영상 제거
9. 2025년 12월 31일 이후 영상 제거
10. 쇼츠(3분 미만) / 라이브 제거
11. 12분 초과 영상 제거
12. 타이틀 #short 포함 제거
13. 분석 제외 채널 제거
14. IQR 1.5 구독자 대비 좋아요 이상치 제거
15. 채널별 드랍률 80% 이상 제거
16. 채널별 영상 50개 미만 제거

## 라이브러리 임포트

In [1]:
import re
import zipfile
from collections import defaultdict
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', None)

In [2]:
def log_drop(step, df, before_v, before_ch):
    """전처리 단계별 드랍 전후 비교 출력 (통일 포맷)"""
    after_ch = df['channel_id'].nunique()
    after_v = len(df)
    print(f"[{step}]")
    print(f"  채널: {before_ch:,}개 → {after_ch:,}개 ({before_ch - after_ch:,}개 제거)")
    print(f"  영상: {before_v:,}개 → {after_v:,}개 ({before_v - after_v:,}개 제거)")

## 데이터 로드

In [3]:
BASE_PATH = Path("../../data")

# 원본 데이터
df_i = pd.read_excel(BASE_PATH / "raw/get_channel_id.xlsx")
df_c = pd.read_excel(BASE_PATH / "raw/channel_info.xlsx")
df_v = pd.read_csv(BASE_PATH / "raw/video_info.csv", sep="\x01")

# 피처 데이터
image_score_df = pd.read_excel(BASE_PATH / "features/image/image_result.xlsx")
channel_feature_df = pd.read_excel(BASE_PATH / "features/concept/channel_feature.xlsx")

C:\Users\Dell5371\AppData\Local\Temp\ipykernel_14808\3644124221.py:6: DtypeWarning: Columns (37,38) have mixed types. Specify dtype option on import or set low_memory=False.
  df_v = pd.read_csv(BASE_PATH / "raw/video_info.csv", sep="\x01")


In [4]:
# 채널별 원본 영상 수 (드랍률 계산용)
orig_video_counts = df_v.groupby('channel_title').size().rename('orig_count')
print(f"원본 채널 수: {len(orig_video_counts)} | 원본 영상 수: {orig_video_counts.sum():,}")

원본 채널 수: 325 | 원본 영상 수: 132,378


## 데이터 전처리

### 수집 이미지 채널 매치

In [5]:
def normalize_name(name):
    """영문자, 숫자, 한글, 일본어만 남기고 모든 특수문자와 공백 제거"""
    return re.sub(r'[^a-zA-Z0-9가-힣ㄱ-ㅎㅏ-ㅣ\u3040-\u309F\u30A0-\u30FF\u4E00-\u9FFF]', '', name)

# zip 파일 내 폴더별 이미지 파일 개수 카운트
zip_path = BASE_PATH / 'features/image/이미지.zip'
folder_file_count = defaultdict(int)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    for file_info in zip_ref.filelist:
        parts = file_info.filename.split('/')
        if len(parts) >= 3 and parts[0] == '이미지' and not file_info.is_dir():
            folder_name = parts[1].strip()
            folder_file_count[folder_name] += 1

df_images = pd.DataFrame([
    {'파일명': folder, '파일갯수': count}
    for folder, count in sorted(folder_file_count.items())
])
print(f"이미지 파일종류: {df_images.shape[0]}")

# 정규화된 이름으로 조인
df_i['채널명_정규화'] = df_i['채널명'].apply(normalize_name)
df_images['파일명_정규화'] = df_images['파일명'].apply(normalize_name)

df_i = df_i.merge(
    df_images[['파일명_정규화', '파일갯수']],
    left_on='채널명_정규화',
    right_on='파일명_정규화',
    how='left'
).drop(columns=['파일명_정규화'])

# 조인 검증
unmatched_images = df_images[~df_images['파일명_정규화'].isin(df_i['채널명_정규화'].unique())]
print(f"조인 실패 df_images: {len(unmatched_images)}개")
print(f"이미지 매치: {df_i['파일갯수'].notna().sum()}행 ({df_i[df_i['파일갯수'].notna()]['channel_id'].nunique()}채널)")
print(f"매치 실패: {df_i['파일갯수'].isna().sum()}행 | channel_id 결측: {df_i['channel_id'].isna().sum()}개")

# 이미지 매치 실패 채널 제거 (매치 성공 행이 하나도 없는 채널만)
matched_ids = set(df_i[df_i['파일갯수'].notna()]['channel_id'].dropna())
exclude_ids = df_i[df_i['파일갯수'].isna() & ~df_i['channel_id'].isin(matched_ids)]['channel_id'].dropna()

before_v, before_ch = len(df_v), df_v['channel_id'].nunique()
df_v = df_v[~df_v['channel_id'].isin(exclude_ids)]
df_c = df_c[df_c['channel_id'].isin(df_v['channel_id'].unique())]
log_drop("이미지 매치 실패 채널 제거", df_v, before_v, before_ch)

이미지 파일종류: 264
조인 실패 df_images: 1개
이미지 매치: 263행 (263채널)
매치 실패: 65행 | channel_id 결측: 0개
[이미지 매치 실패 채널 제거]
  채널: 325개 → 260개 (65개 제거)
  영상: 132,378개 → 105,425개 (26,953개 제거)


### 결과 데이터 조인

In [6]:
# channel_feature_df에서 5개 컬럼을 df_c로 병합
df_c = df_c.merge(
    channel_feature_df[['channel_id', 'sex', 'concept', 'group', 'term', 'platform']],
    on='channel_id',
    how='left'
)
print(f"병합 완료 — df_c: {len(df_c)}행")
print(f"sex 결측: {df_c['sex'].isna().sum()}개")

병합 완료 — df_c: 260행
sex 결측: 48개


In [7]:
# image_score_df의 path에서 채널 폴더명 추출 후 정규화하여 df_c와 조인
# path 예: /content/images/이미지/여르미 YEORUMI/여르미_3.jpg → '여르미 YEORUMI'
image_score_df['채널폴더명'] = image_score_df['path'].str.split('/').str[4]
image_score_df['채널폴더명_정규화'] = image_score_df['채널폴더명'].apply(normalize_name)

# 채널별 final_score 평균 집계
image_agg = image_score_df.groupby('채널폴더명_정규화')['final_score'].mean().reset_index()
image_agg.columns = ['채널폴더명_정규화', 'final_score']

# df_c의 channel_name 정규화
df_c['channel_name_정규화'] = df_c['channel_name'].apply(normalize_name)

# 조인
df_c = df_c.merge(
    image_agg,
    left_on='channel_name_정규화',
    right_on='채널폴더명_정규화',
    how='left'
).drop(columns=['채널폴더명_정규화', 'channel_name_정규화'])

# 검증
print(f"병합 완료 — df_c: {len(df_c)}행")
print(f"final_score 매칭: {df_c['final_score'].notna().sum()}개 | 결측: {df_c['final_score'].isna().sum()}개")
df_c[['channel_name', 'final_score']].head()

병합 완료 — df_c: 260행
final_score 매칭: 253개 | 결측: 7개


,channel_name,final_score
0,Zephyr Archive,NaN
1,유즈하 리코 YUZUHA RIKO,57.712533
2,유우챠 YUUCHA,23.190726
3,유람 Yuram 【PJX】,25.392094
4,율신비,69.147192


In [8]:
# 매칭 실패 채널 확인 — 양쪽 정규화 이름 비교
unmatched = df_c[df_c['final_score'].isna()][['channel_id', 'channel_name']].copy()
unmatched['channel_name_정규화'] = unmatched['channel_name'].apply(normalize_name)

# image_score_df 쪽 폴더명 목록 (정규화 전/후)
image_folders = image_score_df[['채널폴더명', '채널폴더명_정규화']].drop_duplicates().sort_values('채널폴더명_정규화')

print(f"=== 매칭 실패 {len(unmatched)}개 채널 (df_c 쪽) ===")
display(unmatched[['channel_name', 'channel_name_정규화']])

print(f"\n=== image_score_df 폴더명 전체 {len(image_folders)}개 ===")
display(image_folders.reset_index(drop=True))

=== 매칭 실패 7개 채널 (df_c 쪽) ===


,channel_name,channel_name_정규화
0,Zephyr Archive,ZephyrArchive
21,로에 Loe 【V-llage】,로에LoeVllage
28,수잔 헌트 ASMR,수잔헌트ASMR
43,Ryu ch 【 japan custom car 】,Ryuchjapancustomcar
124,BKB CH,BKBCH
135,하루나비,하루나비
181,明楽 レイ /아키라 레이 / Ray Akira 【にじさんじ】,明楽レイ아키라레이RayAkiraにじさんじ



=== image_score_df 폴더명 전체 264개 ===


,채널폴더명,채널폴더명_정규화
0,𝐋𝐈𝐘𝐔 𝐀𝐒𝐌𝐑,
1,AnKe Ch. 안케,AnKeCh안케
2,B_yamu,Byamu
3,Harunabi ch. 하루나비,Harunabich하루나비
4,Jack Ch. 잭,JackCh잭
...,...,...
259,해리 HARRY,해리HARRY
260,허니츄러스 HoneyChurros,허니츄러스HoneyChurros
261,헤비 Hebi,헤비Hebi
262,현단아,현단아


### 채널 정보 조인

In [9]:
# 채널 정보 → df_v에 병합
df_v = df_v.merge(
    df_c[['channel_id', 'subscriber_count', 'video_count', 'sex',
          'concept', 'group', 'term', 'platform', 'final_score']],
    on='channel_id', how='left'
)
print(f"병합 완료 — df_v: {len(df_v):,}행 | 컬럼 수: {df_v.shape[1]}")

병합 완료 — df_v: 105,425행 | 컬럼 수: 49


### 피처 변환

In [10]:
def parse_duration(duration_series):
    """ISO 8601 duration(P1DT1H3M45S, PT3M, P0D 등) -> 초 단위 정수로 변환"""
    def _to_seconds(val):
        if pd.isna(val):
            return 0
        m = re.match(r'P(?:(\d+)D)?(?:T(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?)?', str(val))
        if not m:
            return 0
        d, h, mi, s = (int(x) if x else 0 for x in m.groups())
        return d * 86400 + h * 3600 + mi * 60 + s
    return duration_series.apply(_to_seconds)

def convert_published_to_days(df, reference_date='2026-02-06'):
    """published_at → 기준일로부터 경과 일수(int)로 변환"""
    df = df.copy()
    df['published_at'] = pd.to_datetime(df['published_at'], utc=True)
    df['days_since_published'] = (pd.Timestamp(reference_date, tz='UTC') - df['published_at'].dt.normalize()).dt.days.clip(lower=0)
    return df

df_v['duration'] = parse_duration(df_v['duration'])
df_v = convert_published_to_days(df_v)
print(f"duration 범위: {df_v['duration'].min()} ~ {df_v['duration'].max()}초")
print(f"days_since_published 범위: {df_v['days_since_published'].min()} ~ {df_v['days_since_published'].max()}일")

duration 범위: 0 ~ 63607초
days_since_published 범위: 0 ~ 4562일


### 조회수 필터링

In [11]:
df_v["view_count"] = pd.to_numeric(df_v["view_count"], errors="coerce")

# 조회수 0 제거
before_v, before_ch = len(df_v), df_v['channel_id'].nunique()
df_v = df_v[df_v['view_count'] > 0].reset_index(drop=True)
log_drop("조회수 0 제거", df_v, before_v, before_ch)

# 조회수 100 미만 제거
before_v, before_ch = len(df_v), df_v['channel_id'].nunique()
df_v = df_v[df_v["view_count"] >= 100].reset_index(drop=True)
log_drop("조회수 100 미만 제거", df_v, before_v, before_ch)

# 조회수 1000만 이상 제거
before_v, before_ch = len(df_v), df_v['channel_id'].nunique()
df_v = df_v[df_v["view_count"] < 10_000_000].reset_index(drop=True)
log_drop("조회수 1000만 이상 제거", df_v, before_v, before_ch)

[조회수 0 제거]
  채널: 260개 → 260개 (0개 제거)
  영상: 105,425개 → 105,372개 (53개 제거)
[조회수 100 미만 제거]
  채널: 260개 → 260개 (0개 제거)
  영상: 105,372개 → 102,200개 (3,172개 제거)
[조회수 1000만 이상 제거]
  채널: 260개 → 260개 (0개 제거)
  영상: 102,200개 → 102,179개 (21개 제거)


### 클립채널/부계정 제거

In [ ]:
# 클립 채널 / 부계정 channel_id (수동 지정)
exclude_manual = {
    "UCgGvSg2lscdNUx9ZJIBh9FQ",
    "UC3JYpe9DEFydDKtLpsHGbxA",
    "UC4joLChHaxuld3elqjAjhRw",
    "UCrKxP6IDEsFLuX_g8bx-43w",
    "UCS7ppVm16nATTo9S---upPQ",
    "UCUgGNomn6nUX7_YDlVjRnnA",
    "UC9yLAGEMe-w2LniDmr3Zjdw",
    "UC1CalxlthKnoIedtZRlOLTA",
    "UCFP9UkgIM_U8NfzRbYEOQdA",
    "UCXnKI_Ak-XYkXoCD_JS1YbQ",
    "UCj2qZQT1oF2MULq2MZMqu0w",
    "UC7hffDQLKIEG-_zoAQkMIvg",
    "UCIBGKlMbLjVJX7TvVuxbSHg",
    "UCaOr7NH-86RqjFL3_dymfzg",
    "UCj6adKvatGoaDTrnBfeLjKw",
    "UCGrcpQTFZ7xMdTOycqDuajQ",
    "UCTZ3T55Vk6bB7yxmukz_MnA",
    "UCkmzHPB67JpEf2S5Ux9bfDg",
    "UCaacwbLBfcQDeazRwQ2dApg",
    "UC2BYEZCRaFXUD5Q81Qc8dVw",
    "UCq2WDoKo7otLhMBIK0CqyuA",
}

# 해당 채널을 df_v / df_c에서 제거
before_v, before_ch = len(df_v), df_v['channel_id'].nunique()
df_v = df_v[~df_v['channel_id'].isin(exclude_manual)].reset_index(drop=True)
df_c = df_c[~df_c['channel_id'].isin(exclude_manual)].reset_index(drop=True)
log_drop("클립채널/부계정 제거", df_v, before_v, before_ch)

### 활동 중단 필터링
- 6개월 이상 활동이 없으면 수익창출 중단 가능

In [ ]:
# 채널별 가장 최근 영상이 180일(6개월) 이상인 채널
inactive = df_v.groupby(['channel_id', 'channel_title']).agg(
    latest_days_ago=('days_since_published', 'min'),
    video_count=('video_id', 'size')
).reset_index()
inactive = inactive[inactive['latest_days_ago'] >= 180].sort_values('latest_days_ago', ascending=False)
inactive_ids = inactive['channel_id'].tolist()

# 해당 채널을 df_v / df_c에서 제거
before_v, before_ch = len(df_v), df_v['channel_id'].nunique()
df_v = df_v[~df_v['channel_id'].isin(inactive_ids)].reset_index(drop=True)
df_c = df_c[~df_c['channel_id'].isin(inactive_ids)].reset_index(drop=True)
log_drop("활동 중단 채널 제거 (6개월)", df_v, before_v, before_ch)

### 2022년 이전 영상 제거

In [ ]:
# 2022-01-01 이전 업로드 영상 제거
before_v, before_ch = len(df_v), df_v['channel_id'].nunique()
df_v = df_v[df_v['published_at'] >= '2022-01-01'].reset_index(drop=True)
log_drop("2022년 이전 영상 제거", df_v, before_v, before_ch)

### 2025년 12월 31일 이후 영상 제거

In [ ]:
# 2025-12-31 이후 (2026-01-01 이상) 업로드 영상 제거
before_v, before_ch = len(df_v), df_v['channel_id'].nunique()
df_v = df_v[df_v['published_at'] < '2026-01-01'].reset_index(drop=True)
log_drop("2025년 12월 31일 이후 영상 제거", df_v, before_v, before_ch)

### 쇼츠(3분 미만) / 라이브 제거

In [16]:
# 3분 미만 영상 제거
before_v, before_ch = len(df_v), df_v['channel_id'].nunique()
df_v = df_v[df_v['duration'] >= 180].reset_index(drop=True)
log_drop("3분 미만 영상 제거", df_v, before_v, before_ch)

# 라이브 영상 제거
before_v, before_ch = len(df_v), df_v['channel_id'].nunique()
df_v = df_v[df_v['is_live_content'] != True].reset_index(drop=True)
log_drop("라이브 영상 제거", df_v, before_v, before_ch)

[3분 미만 영상 제거]
  채널: 211개 → 210개 (1개 제거)
  영상: 75,556개 → 37,823개 (37,733개 제거)
[라이브 영상 제거]
  채널: 210개 → 209개 (1개 제거)
  영상: 37,823개 → 27,373개 (10,450개 제거)


### 영상길이 12분 초과 제거

In [ ]:
# 12분(720초) 초과 영상 제거
before_v, before_ch = len(df_v), df_v['channel_id'].nunique()
df_v = df_v[df_v["duration"] < 720].reset_index(drop=True)
log_drop("12분 초과 영상 제거", df_v, before_v, before_ch)

### 타이틀 #short 포함 제거

In [ ]:
# 타이틀에 #short 포함된 영상 제거
before_v, before_ch = len(df_v), df_v['channel_id'].nunique()
df_v = df_v[~df_v["title"].str.contains(r"#short", case=False, na=False)].reset_index(drop=True)
log_drop("타이틀 #short 제거", df_v, before_v, before_ch)

### 분석 제외 채널 제거

In [ ]:
# 분석 제외 채널
drop_channels = [
    "UCYkp5UQqqfkWYkWm_jJkA_A",  # 도롱챠 DOLONGCHA
    "UCuoLHlnnu64Cn66VTGH4WEw",  # 송찐빵
    "UCb4MU4hP3BgaRDsnVW-J-QA",  # 늦잠【싸이코드】
    "UCNpcMizRJ73OJm48LszuytQ",  # MaWang 마왕
    "UC0QDb4XkScbRzNNQWnIywjg",  # 하루토 【싸이코드】
    "UC9UBS38T2wKKbEKgPFdAoFQ",  # 여르미 YEORUMI
    "UCDZTV0gSwZBZriB3GmnGq0w",  # Myoya Ch. 묘야
    "UChEejxuuRu15AaVBV5kkM0g",  # 소히SOHEE
    "UCfCKPSq95ekDxpZB0kY6rWg"   # 루민 Lumin
]

# 해당 채널을 df_v / df_c에서 제거
before_v, before_ch = len(df_v), df_v['channel_id'].nunique()
df_v = df_v[~df_v['channel_id'].isin(drop_channels)].reset_index(drop=True)
df_c = df_c[~df_c['channel_id'].isin(drop_channels)].reset_index(drop=True)
log_drop("분석 제외 채널 제거", df_v, before_v, before_ch)

### IQR 1.5 구독자수 대비 좋아요수 이상치 제거

In [20]:
# 구독자수 대비 좋아요 비율 생성
df_v["like_count"] = pd.to_numeric(df_v["like_count"], errors="coerce")
df_v["subscriber_count"] = pd.to_numeric(df_v["subscriber_count"], errors="coerce")
df_v["like_per_sub"] = df_v["like_count"] / df_v["subscriber_count"].replace(0, float("nan"))

# 채널별 IQR 기반 이상치 제거 (k=1.5)
k = 1.5
q1 = df_v.groupby("channel_id")["like_per_sub"].transform("quantile", 0.25)
q3 = df_v.groupby("channel_id")["like_per_sub"].transform("quantile", 0.75)
iqr = q3 - q1

before_v, before_ch = len(df_v), df_v['channel_id'].nunique()
df_v = df_v[(df_v["like_per_sub"] >= q1 - k * iqr) & (df_v["like_per_sub"] <= q3 + k * iqr)].reset_index(drop=True)
log_drop("IQR 1.5 이상치 제거 (구독자 대비 좋아요)", df_v, before_v, before_ch)

[IQR 1.5 이상치 제거 (구독자 대비 좋아요)]
  채널: 198개 → 198개 (0개 제거)
  영상: 20,101개 → 18,715개 (1,386개 제거)


### 채널별 영상 드랍률 기반 필터링

In [ ]:
DROP_THRESHOLD = 80

# 채널별 원본 대비 현재 영상 수로 드랍률(%) 계산
current_counts = df_v.groupby('channel_title').size().rename('current_count')
drop_summary = pd.DataFrame({
    'orig_count': orig_video_counts,
    'current_count': current_counts,
}).reindex(orig_video_counts.index)

drop_summary['current_count'] = drop_summary['current_count'].fillna(0).astype(int)
drop_summary['drop_pct'] = ((drop_summary['orig_count'] - drop_summary['current_count']) / drop_summary['orig_count'] * 100).round(1)
drop_summary = drop_summary.sort_values('drop_pct', ascending=False)

# display(drop_summary)

# 드랍률이 임계값(DROP_THRESHOLD) 이상인 채널 제거
high_drop = drop_summary[drop_summary['drop_pct'] >= DROP_THRESHOLD]
high_drop_titles = high_drop.index.tolist()

before_v, before_ch = len(df_v), df_v['channel_id'].nunique()
df_v = df_v[~df_v['channel_title'].isin(high_drop_titles)].reset_index(drop=True)
df_c = df_c[~df_c['channel_name'].isin(high_drop_titles)].reset_index(drop=True)
log_drop(f"채널별 드랍률 {DROP_THRESHOLD}% 이상 제거", df_v, before_v, before_ch)

### 채널의 영상 개수 50개 미만인 채널 드랍

In [ ]:
MIN_VIDEOS = 50

# 보유 영상 수가 MIN_VIDEOS 미만인 채널 추출
channel_video_counts = df_v.groupby('channel_id')['video_id'].count()
small_channels = channel_video_counts[channel_video_counts < MIN_VIDEOS].index

# 해당 채널을 df_v / df_c에서 제거
before_v, before_ch = len(df_v), df_v['channel_id'].nunique()
df_v = df_v[~df_v['channel_id'].isin(small_channels)].reset_index(drop=True)
df_c = df_c[~df_c['channel_id'].isin(small_channels)].reset_index(drop=True)
log_drop(f"채널별 영상 {MIN_VIDEOS}개 미만 제거", df_v, before_v, before_ch)

### 결과 저장

In [ ]:
# 전처리 완료 데이터 저장
df_v.to_csv(BASE_PATH / "processed/video_cleaned.csv", sep="\x01", index=False)
print(f"저장 완료 — {len(df_v):,}행 | {df_v['channel_id'].nunique()}채널")